# Intro to Python & Supercomputing on Frontera
### PFX Pre-Freshman Program — 1-hour workshop

Welcome! For the next hour you're going to write real Python code **and** run it on
Frontera, a research supercomputer at the Texas Advanced Computing Center (TACC).

**How this works:** each gray box below is a *cell*. Click a cell, then press
**`Shift` + `Enter`** to run it. Run the cells in order, top to bottom. Change things,
break things, re-run them. That's how you learn.

---
**Roadmap for the hour**
1. Where am I? (you're already on a supercomputer) — 5 min
2. Python basics — 20 min
3. Why a supercomputer? Doing many things at once — 20 min
4. The rest of Frontera: files, modules, and jobs — 5 min
5. Wrap-up & where to go next — 5 min

## Part 1 — Where am I?

You're running this notebook through the **TACC Analysis Portal (TAP)**, which started
Jupyter for you on a **compute node** of **Frontera**.

Frontera is a research supercomputer at the Texas Advanced Computing Center (TACC) in
Austin, Texas, funded by the National Science Foundation. It is built from **8,368 compute
nodes**. Each node has **56 processor cores** and **192 GB of memory**.

Let's prove where we are. Run the cell below.

In [ ]:
# The '!' lets us run a terminal command from inside Python.
# 'hostname' prints the name of the machine we're actually running on.
!hostname

That name (something like `c123-456`) is one node deep inside Frontera's machine room in
Austin, Texas. Now let's see the scale of the machine you're plugged into.

In [ ]:
import os

node = os.cpu_count()

print(f"This node you're on:     {node} cores")
print(f"Reserved for us today:   250 nodes  = {250 * node:,} cores")
print(f"All of Frontera:         8,368 nodes = {8368 * node:,} cores")

The power of a supercomputer is **scale**: thousands of nodes that can work together, so a
single job can run across hundreds of them at once.

Today we're working on **one node**, 56 cores. Even that is enough to feel the difference, as
you'll see in Part 3.

## Part 2 — Python basics

Python is one of the most popular programming languages in science. Let's learn the
pieces you'll use constantly. Run every cell, then try changing the values.

### 2.1 Printing and variables

A **variable** is a labeled box that holds a value. `=` puts a value in the box.

In [ ]:
name = "your name here"   # <-- change this to YOUR name, then re-run (Shift+Enter)
school = "Morehouse"

print("Hi", name + "!")
print("Welcome to", school)

### 2.2 Numbers and math

Python is a powerful calculator. Try changing the numbers.

In [ ]:
a = 12
b = 5

print("add:     ", a + b)
print("subtract:", a - b)
print("multiply:", a * b)
print("divide:  ", a / b)
print("power:   ", a ** b)   # a to the power of b

### 2.3 Text (strings)

Text values are called **strings**. An **f-string** (note the `f` before the quote) lets
you drop variables right into a sentence.

In [ ]:
first = "Ada"
last = "Lovelace"
age = 17

message = f"{first} {last} is {age} years old."
print(message)
print(f"In 5 years she'll be {age + 5}.")

### 2.4 Lists

A **list** holds many values in order. We count positions starting at **0**.

In [ ]:
snacks = ["chips", "cookies", "fruit", "water"]

print("The whole list:", snacks)
print("How many items:", len(snacks))
print("First item:", snacks[0])
print("Last item:", snacks[-1])

snacks.append("popcorn")   # add something new to the end
print("After adding:", snacks)

### 2.5 Loops

A **loop** repeats an action for every item. This is where computers beat humans: they
never get bored doing the same thing a million times.

In [ ]:
for snack in snacks:
    print("I like", snack)

In [ ]:
# range(n) counts 0, 1, 2, ... up to (but not including) n
total = 0
for number in range(1, 101):   # 1 through 100
    total = total + number

print("The sum of 1 to 100 is", total)

### 2.6 Making decisions (`if`)

Programs make choices with `if` / `elif` / `else`. Change `score` and re-run.

In [ ]:
score = 85

if score >= 90:
    print("Grade: A")
elif score >= 80:
    print("Grade: B")
elif score >= 70:
    print("Grade: C")
else:
    print("Keep going!")

### 2.7 Functions

A **function** is a reusable recipe. You define it once, then *call* it as many times as
you want. `def` starts the recipe; `return` hands back the answer.

In [ ]:
def celsius_to_fahrenheit(c):
    f = c * 9 / 5 + 32
    return f

print("0C  =", celsius_to_fahrenheit(0),  "F")
print("25C =", celsius_to_fahrenheit(25), "F")
print("37C =", celsius_to_fahrenheit(37), "F")

### 2.8 You try it

Write a function `square` that returns a number times itself, then print the squares of
1 through 5. A starting point is below — fill in the blank and run it.

In [ ]:
def square(n):
    return  # <-- return n times n

for i in range(1, 6):
    print(i, "squared is", square(i))

## Part 3 — Why a supercomputer? Doing many things at once

So far every cell ran on **one core**. A supercomputer's real trick is doing many things
**at the same time** across many cores. This is called **parallel computing**.

We'll try it on two real jobs: **editing a batch of photos**, and **modeling how a disease
spreads**. Each time we'll run the job the slow way (1 core), then the fast way (all the
node's cores), and time the difference.

First, a helper so our parallel code works smoothly:

In [ ]:
import os, time
import multiprocessing as mp

# Use "fork" so functions we define here work with multiple processes.
try:
    mp.set_start_method("fork")
except RuntimeError:
    pass
from multiprocessing import Pool

print("This node has", os.cpu_count(), "cores to work with.")

### 3.1 Example one: editing a batch of photos  📷

Imagine you just took **hundreds of photos** and want to apply the same filter (a blur) to
all of them. Editing one photo is quick. Editing hundreds, one at a time, is slow.

But each photo is independent, so we can hand a different photo to each core and edit them
**all at once**. This is exactly how real photo, video, and science-imaging pipelines work.

First, the code that makes an image and blurs it. Just run this cell.

In [ ]:
import numpy as np

IMG_SIZE = 256   # width and height of each image, in pixels
BLUR_PASSES = 20 # how many times to blur (more = stronger filter, more work)

def make_image(seed):
    """Create a colorful image full of overlapping circles."""
    rng = np.random.default_rng(seed)
    img = np.zeros((IMG_SIZE, IMG_SIZE, 3))
    yy, xx = np.ogrid[:IMG_SIZE, :IMG_SIZE]
    for _ in range(6):
        cy, cx = rng.integers(0, IMG_SIZE), rng.integers(0, IMG_SIZE)
        r = rng.integers(20, 80)
        img[(yy - cy) ** 2 + (xx - cx) ** 2 < r * r] = rng.random(3)
    return img

def blur_once(img):
    """Blur an image a little by averaging each pixel with its neighbors."""
    o = img.copy(); o[1:-1] = (img[:-2] + img[1:-1] + img[2:]) / 3
    p = o.copy(); p[:, 1:-1] = (o[:, :-2] + o[:, 1:-1] + o[:, 2:]) / 3
    return p

def edit_one_photo(seed):
    """Make one image and apply the blur filter to it."""
    img = make_image(seed)
    for _ in range(BLUR_PASSES):
        img = blur_once(img)
    return float(img.mean())   # return a tiny summary, not the whole image

print("Ready to edit photos.")

Let's *see* what the filter does. This shows one image before and after the blur.

In [ ]:
import matplotlib.pyplot as plt

original = make_image(0)
filtered = original.copy()
for _ in range(BLUR_PASSES):
    filtered = blur_once(filtered)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(np.clip(original, 0, 1)); ax[0].set_title("original");  ax[0].axis("off")
ax[1].imshow(np.clip(filtered, 0, 1)); ax[1].set_title("filtered"); ax[1].axis("off")
plt.show()

Now the real test. We'll edit **a whole batch** of photos.

**The slow way first:** one core, one photo at a time.

In [ ]:
NUM_PHOTOS = 500   # <-- the size of our photo shoot

photos = list(range(NUM_PHOTOS))

start = time.time()
for seed in photos:
    edit_one_photo(seed)
photo_serial = time.time() - start

print(f"Edited {NUM_PHOTOS} photos on 1 core in {photo_serial:.1f} seconds.")

**Now the fast way:** hand the photos out across *all* the node's cores at once.

In [ ]:
start = time.time()
with Pool(os.cpu_count()) as pool:
    pool.map(edit_one_photo, photos)
photo_parallel = time.time() - start

print(f"Edited {NUM_PHOTOS} photos on {os.cpu_count()} cores in {photo_parallel:.1f} seconds.")
print(f"That's about {photo_serial / photo_parallel:.0f}x faster, for the exact same work.")

### 3.2 Example two: modeling how a disease spreads  🦠

Public-health researchers ask "what if?" a lot. What if a disease spreads faster? What if
people recover sooner? Each "what if" is one **simulation**, and to understand the whole
picture they run **thousands** of them. That's a perfect job for many cores.

Our simple model tracks three groups over time: people who are **S**usceptible,
**I**nfected, and **R**ecovered. Two numbers control it:
- `beta` — how easily the disease spreads
- `gamma` — how quickly people recover

Run this cell to define the model.

In [ ]:
def peak_infected(settings):
    """Run one epidemic scenario. Return the largest fraction infected at once."""
    beta, gamma = settings
    S, I, R = 0.999, 0.001, 0.0     # start: almost everyone susceptible
    worst = I
    for day in range(160):
        new_infections = beta * S * I
        new_recoveries = gamma * I
        S -= new_infections
        I += new_infections - new_recoveries
        R += new_recoveries
        if I > worst:
            worst = I
    return worst

print("One scenario, beta=0.3, gamma=0.1 -> peak infected:",
      f"{peak_infected((0.3, 0.1)) * 100:.1f}% of people at once")

Now we sweep across **thousands** of combinations of `beta` and `gamma`, one scenario per
grid point. **The slow way first:** one core.

In [ ]:
import numpy as np

GRID = 400   # <-- we test GRID x GRID combinations

betas  = np.linspace(0.10, 0.50, GRID)
gammas = np.linspace(0.05, 0.40, GRID)
scenarios = [(b, g) for b in betas for g in gammas]

start = time.time()
results = [peak_infected(s) for s in scenarios]
sweep_serial = time.time() - start

print(f"Ran {len(scenarios):,} scenarios on 1 core in {sweep_serial:.1f} seconds.")

**Now across all the node's cores:**

In [ ]:
start = time.time()
with Pool(os.cpu_count()) as pool:
    results = pool.map(peak_infected, scenarios)
sweep_parallel = time.time() - start

print(f"Ran {len(scenarios):,} scenarios on {os.cpu_count()} cores in {sweep_parallel:.1f} seconds.")
print(f"That's about {sweep_serial / sweep_parallel:.0f}x faster.")

Every one of those scenarios is a dot on this map. Bright areas are outbreaks where a large
share of people are infected at the same time. This is the kind of picture that would take
ages to build one scenario at a time.

In [ ]:
peaks = np.array(results).reshape(GRID, GRID) * 100

plt.figure(figsize=(6, 5))
plt.imshow(peaks, origin="lower", aspect="auto", cmap="inferno",
           extent=[gammas[0], gammas[-1], betas[0], betas[-1]])
plt.colorbar(label="peak % infected at once")
plt.xlabel("recovery rate  (gamma)")
plt.ylabel("infection rate  (beta)")
plt.title(f"{GRID * GRID:,} epidemic scenarios")
plt.show()

### What just happened

Both jobs did the **same total work** whether on 1 core or many. Spreading the work across
the node's cores just finished it far sooner.

And remember the scale from Part 1: that was **one node**. We reserved **250 nodes** for this
room today, and Frontera has **8,368**. Real research jobs run across hundreds of them at
once, which is how scientists edit millions of images, or run millions of simulations, in the
time it took us to do a few thousand.

## Part 4 — The rest of Frontera

You won't master this today, but here's the vocabulary so it's familiar later.

**Login nodes vs. compute nodes.** When researchers connect to Frontera they first land on a
shared *login node*, fine for editing files, not for heavy computing. Real work runs on
*compute nodes* like the one running this notebook right now.

**Filesystems** — three places to keep files:

| Name | Use it for |
|------|-----------|
| `$HOME`    | small stuff: code, settings (25 GB) |
| `$WORK`    | your project files (1 TB) |
| `$SCRATCH` | big temporary job data (cleared periodically) |

Run the cell to see your own folders:

In [ ]:
!echo "HOME    is:" $HOME
!echo "WORK    is:" $WORK
!echo "SCRATCH is:" $SCRATCH

**Jobs and the scheduler (Slurm).** Frontera is shared by thousands of researchers, so you
don't just grab nodes whenever you want. You describe the job you need and it goes into a
**queue**; a program called **Slurm** hands you nodes when they're free. TAP did exactly this
for you to open this notebook, using our **Morehouse** reservation so you didn't have to wait.

See what's running for you right now:

In [ ]:
!squeue -u $USER

## Part 5 — Wrap-up

In one hour you:
- ran code on a research supercomputer,
- learned Python variables, math, strings, lists, loops, `if`, and functions,
- used many cores at once to edit a batch of photos and model thousands of epidemics,
- and saw that the real power of HPC is **scale**: many cores, and many nodes, working together.

**Keep going:**
- Frontera user guide: https://docs.tacc.utexas.edu/hpc/frontera/
- TACC Analysis Portal: https://tap.tacc.utexas.edu
- Free Python practice: https://www.python.org and https://www.w3schools.com/python/

### Challenge (if you have time)
Below, write a function `is_even(n)` that returns `True` when a number is even and `False`
when it's odd. (Hint: a number is even when `n % 2 == 0`.) Then loop 1 through 10 and print
each number with whether it's even.

In [ ]:
def is_even(n):
    return  # <-- your code here

# your loop here

---
*Great work. You're officially a supercomputer user.* 🚀